# Telecom Churn Analysis — Experimental Notebook

This notebook runs the **full ML pipeline** using the production `src/` modules.

**Pipeline:** `load_data → validate → clean_data → feature_engineering → train → evaluate → infer`

In [11]:
import sys
from pathlib import Path

# Ensure the repo root is on sys.path so `src` is importable
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

Repo root: e:\MLops\1-mlops-kickoff-repo


In [12]:
import pandas as pd
import matplotlib.pyplot as plt

from src.load_data import load_data
from src.validate import validate_dataframe
from src.clean_data import clean_data
from src.feature_engineering import build_features, FeatureConfig
from src.train import train_model
from src.evaluate import evaluate_model
from src.infer import run_inference
from sklearn.preprocessing import FunctionTransformer

## 1. Load Data

In [13]:
raw_path = REPO_ROOT / "data" / "raw" / "telecom_churn.csv"
df = load_data(raw_path)
print(f"Shape: {df.shape}")
df.head()

2026-03-09 18:14:58,957 - INFO - Loaded 'e:\MLops\1-mlops-kickoff-repo\data\raw\telecom_churn.csv': 3333 rows, 11 columns


Shape: (3333, 11)


,Churn,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins
0,0,128,1,1,2.7,1,265.1,110,89.0,9.87,10.0
1,0,107,1,1,3.7,1,161.6,123,82.0,9.78,13.7
2,0,137,1,0,0.0,0,243.4,114,52.0,6.06,12.2
3,0,84,0,0,0.0,2,299.4,71,57.0,3.10,6.6
4,0,75,0,0,0.0,3,166.7,113,41.0,7.42,10.1


## 2. Validate Raw Data

In [14]:
REQUIRED_COLUMNS = [
    "AccountWeeks", "DataUsage", "CustServCalls",
    "DayMins", "DayCalls", "MonthlyCharge",
    "OverageFee", "RoamMins", "Churn",
    "ContractRenewal", "DataPlan",
]

validate_dataframe(df, REQUIRED_COLUMNS)
print("Validation passed.")

2026-03-09 18:14:58,968 - INFO - Starting data validation checks...
2026-03-09 18:14:58,968 - INFO - DataFrame shape: (3333, 11)
2026-03-09 18:14:58,968 - INFO - All required columns are present.
2026-03-09 18:14:58,969 - INFO - No missing values in required columns.
2026-03-09 18:14:58,971 - INFO - No duplicate rows found.
2026-03-09 18:14:58,972 - INFO - Telecom numeric columns are non-negative.
2026-03-09 18:14:58,974 - INFO - Telecom binary columns contain only 0/1.
2026-03-09 18:14:58,974 - INFO - All validation checks passed.


Validation passed.


## 3. Clean Data

In [15]:
df_clean = clean_data(df)
print(f"Shape after cleaning: {df_clean.shape}")
df_clean.head()

Shape after cleaning: (3333, 11)


,churn,accountweeks,contractrenewal,dataplan,datausage,custservcalls,daymins,daycalls,monthlycharge,overagefee,roammins
0,0,128,1,1,2.7,1,265.1,110,89.0,9.87,10.0
1,0,107,1,1,3.7,1,161.6,123,82.0,9.78,13.7
2,0,137,1,0,0.0,0,243.4,114,52.0,6.06,12.2
3,0,84,0,0,0.0,2,299.4,71,57.0,3.10,6.6
4,0,75,0,0,0.0,3,166.7,113,41.0,7.42,10.1


## 4. Feature Engineering

In [16]:
TARGET_COL = "churn"

cfg = FeatureConfig(
    target_col=TARGET_COL,
    numeric_cols=(
        "accountweeks", "datausage", "custservcalls",
        "daymins", "daycalls", "monthlycharge",
        "overagefee", "roammins",
    ),
    categorical_cols=("contractrenewal", "dataplan"),
)

df_feat = build_features(df_clean, cfg)
print(f"Shape after feature engineering: {df_feat.shape}")
df_feat.head()

Shape after feature engineering: (3333, 23)


,churn,accountweeks,datausage,custservcalls,daymins,daycalls,monthlycharge,overagefee,roammins,accountweeks_is_missing,...,daycalls_is_missing,monthlycharge_is_missing,overagefee_is_missing,roammins_is_missing,contractrenewal_0.0,contractrenewal_1.0,contractrenewal_nan,dataplan_0.0,dataplan_1.0,dataplan_nan
0,0,128,2.7,1,265.1,110,89.0,9.87,10.0,0,...,0,0,0,0,False,True,False,False,True,False
1,0,107,3.7,1,161.6,123,82.0,9.78,13.7,0,...,0,0,0,0,False,True,False,False,True,False
2,0,137,0.0,0,243.4,114,52.0,6.06,12.2,0,...,0,0,0,0,False,True,False,True,False,False
3,0,84,0.0,2,299.4,71,57.0,3.10,6.6,0,...,0,0,0,0,True,False,False,True,False,False
4,0,75,0.0,3,166.7,113,41.0,7.42,10.1,0,...,0,0,0,0,True,False,False,True,False,False


## 5. Train Model

In [17]:
y = df_feat[TARGET_COL]
X = df_feat.drop(columns=[TARGET_COL])

preprocessor = FunctionTransformer()  # features already engineered
MODEL_PATH = str(REPO_ROOT / "models" / "model.pkl")
PROBLEM_TYPE = "classification"

fitted_pipeline, X_val, y_val, X_test, y_test = train_model(
    X, y, preprocessor, PROBLEM_TYPE, MODEL_PATH
)
print(f"Test set size: {len(X_test)}")
print(f"Validation set size: {len(X_val)}")

2026-03-09 18:14:59,014 - INFO - Config file not found at config.yaml. Using function defaults.
2026-03-09 18:14:59,014 - INFO - Starting training: problem_type=classification
2026-03-09 18:14:59,015 - INFO - Splitting data (3-way): test_size=0.200 val_size=0.200 random_state=42
2026-03-09 18:14:59,019 - INFO - Split sizes: train=1999 val=667 test=667
2026-03-09 18:14:59,019 - INFO - Fitting pipeline on training split only...
2026-03-09 18:14:59,029 - INFO - Saved model artifact to e:\MLops\1-mlops-kickoff-repo\models\model.pkl


Test set size: 667
Validation set size: 667


## 6. Evaluate Model

In [18]:
metric = evaluate_model(fitted_pipeline, X_test, y_test, PROBLEM_TYPE)
print(f"\nWeighted F1 Score: {metric:.4f}")

2026-03-09 18:14:59,035 - INFO - Config file not found at config.yaml. Using defaults.
2026-03-09 18:14:59,053 - INFO - Starting evaluation: problem_type=classification
2026-03-09 18:14:59,079 - INFO - Classification metric: f1_weighted=0.826593
2026-03-09 18:14:59,083 - INFO - Classification report:
              precision    recall  f1-score   support

           0       0.87      0.98      0.92       570
           1       0.55      0.18      0.27        97

    accuracy                           0.86       667
   macro avg       0.71      0.58      0.59       667
weighted avg       0.83      0.86      0.83       667

2026-03-09 18:14:59,264 - INFO - Saved confusion matrix to reports\figures\confusion_matrix.png



Weighted F1 Score: 0.8266


In [19]:
# Show the confusion matrix plot that evaluate_model saved
cm_path = REPO_ROOT / "reports" / "figures" / "confusion_matrix.png"
if cm_path.exists():
    img = plt.imread(str(cm_path))
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Confusion Matrix (saved by evaluate_model)")
    plt.show()

C:\Users\Usuario\AppData\Local\Temp\ipykernel_10512\2573591362.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Inference

In [20]:
predictions = run_inference(fitted_pipeline, X_test)
print(f"Predictions shape: {predictions.shape}")
predictions.head(10)

2026-03-09 18:14:59,289 - INFO - Running inference using model.predict
2026-03-09 18:14:59,291 - INFO - churn_probability computed (min=0.0025, max=0.9762)
2026-03-09 18:14:59,293 - INFO - Example mapped labels (first 3): ['No Churn', 'No Churn', 'No Churn']
2026-03-09 18:14:59,296 - INFO - Predictions saved -> E:\MLops\1-mlops-kickoff-repo\data\inference\predictions.csv


Predictions shape: (667, 1)


,prediction
601,0
2050,0
3200,0
1953,0
1119,0
2204,0
1888,0
738,0
3087,0
539,0


## 8. Summary

The full production pipeline ran end-to-end using the `src/` modules:

| Step | Module | Status |
|------|--------|--------|
| Load | `load_data.py` | Done |
| Validate | `validate.py` | Done |
| Clean | `clean_data.py` | Done |
| Features | `feature_engineering.py` | Done |
| Train | `train.py` | Done |
| Evaluate | `evaluate.py` | Done |
| Infer | `infer.py` | Done |